<a href="https://colab.research.google.com/github/Jayadeep19/pytorch/blob/master/6.%20Attention%20mechanism/1.%20Attention%20basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Attention basics

- In this notebook, I will implement attention mechanism from scratch using pytorch. I would not be explaining how attention works.
- I have written a blog about the attention and how it is the backbone in transformers. See [attention](https://jayadeep19.github.io/til-notes/transformers)

- First we will see a simple implementation of Self attention
- and then I will try to apply the attention to MNIST dataset to see how Query, Key, Value and embedding dimensions effect each other!!

## Simple Self attention:
- we use a simple sentence to generate a context aware embedding using query, key and value vectors.

1. assign each token to unique Id
2. Embed each token using pytorchs embedding layer
3. Create Query, Key and Value vectors
4. Calculate attention scores
5. calculate attention weights (softmax application)
6. Generate Context aware Vectors using Value vectors!!

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# Data prep.
sequence = 'Introduction to attention mechanism'

# we will first assign each word to a unique index
tokens = sorted(sequence.split())

idx = {word: i for i, word in enumerate(tokens)}
print(idx)

{'Introduction': 0, 'attention': 1, 'mechanism': 2, 'to': 3}


In [4]:
#convert to pytorch tensor
temp_list = [idx[i] for i in sequence.split()]
seq_ten = torch.tensor(temp_list)
print(seq_ten)

tensor([0, 3, 1, 2])


*- embed the sequence using [nn.Embedding](https://towardsdatascience.com/the-secret-to-improved-nlp-an-in-depth-look-at-the-nn-embedding-layer-in-pytorch-6e901e193e16/)*

In [8]:
vocab = 50 # the vocabulary size, ex: 50 words in vocab
# pytorch allocates len(vocab) rows in the lookuptable (weights!!). One row per possible token!!

ndim = 3 # I want to represent each token with a 3D embedding

# before initiating the embedding layer, we can also set seed to some number for reproducability
# (weights are initiated randomly)

embed_lay =nn.Embedding(vocab, ndim)

embedded_sentence = embed_lay(seq_ten)
print(embedded_sentence.detach()) #auto grad off
print(f'shape of the embedding: {embedded_sentence.shape}')

tensor([[ 1.1264,  0.2353, -0.0318],
        [ 0.3066,  0.7612, -0.2978],
        [ 0.2854, -0.8796, -2.3534],
        [-0.0436,  0.0273,  1.1626]])
shape of the embedding: torch.Size([4, 3])


- each row represents a token in our sequence.
- the embedding shape is [4,3] representing 4 tokens of 3D vectors.

*- self attention*
- The dimensions of W_q should be \[3, \<dim\>\] (while '3' is embed_sent.shpae[1])
  - since emb_sent @ W_q = Query of shape \[4,\<dim\>\]
- similarly W_v is \[3, \<dim\>\]
  -since emb_sen @ W_v = Key of shape \[4,\dim\>\]
  - and Query @ Key.T = attention scores
- Then value dim automatically becomes \[\<dim\>,\<dim\>\]
  - since attention scores @ value

- The shapes can be changed based on how we do the multiplications
  - **either** embedded_sentence @ W_query (as in this example)
  - **or** W_query @ embedded_sentence then the shapes become:
  ![or](https://miro.medium.com/v2/resize:fit:720/format:webp/1*8xqD4AYTwn6mQXNw0uhDCg.gif)
  - easy to calculate.

In [13]:
#get the embed dim the last one (3)
e_d = embedded_sentence.shape[1]

#sample end dim for query (3,4), key(3,4), value(4,4)
#put other dim and try to compute the end dim of key.
q_d, k_d, v_d = 4,4,4

#initiate weights (normally if we are defining the weights as a parameter in a network,
# then we have to specify that these are nn.Parameter() for autograd computation)
W_query = torch.randn(e_d, q_d)
W_key = torch.randn(e_d, k_d)
W_value = torch.randn(e_d, v_d)

#compute query, Key, Value
query = embedded_sentence @ W_query
key = embedded_sentence @ W_key
value = embedded_sentence @ W_value

print(f'shape of the query matrix: {query.shape}')
print(f'shape of the key matrix: {key.shape}')
print(f'shape of the value matrix: {value.shape}')


shape of the query matrix: torch.Size([4, 4])
shape of the key matrix: torch.Size([4, 4])
shape of the value matrix: torch.Size([4, 4])


In [21]:
# calculate attention scores (Q @ K.T)

attention_scores = query @ key.T
attention_scores = attention_scores/ torch.sqrt(torch.tensor(k_d))
print(attention_scores.detach(), '\n', attention_scores.shape)

#convert the scores to weights by applying softmax
attention_weights = F.softmax(attention_scores, dim = 1)   #dim 0 is columns, 1 is rows.
print(attention_weights.detach())

#we can check if the scores of each row is 1 or not
print(f'score sum of row 1 is : {attention_weights[0].sum()}')

tensor([[ 0.6745,  1.0436, -3.7538,  1.2087],
        [ 1.2005, -1.3405,  0.1009,  0.9202],
        [ 3.0763, -0.4001, -4.8286,  2.9459],
        [-1.8121,  1.1021,  1.2212, -1.4940]]) 
 torch.Size([4, 4])
tensor([[2.4013e-01, 3.4732e-01, 2.8659e-03, 4.0969e-01],
        [4.6139e-01, 3.6354e-02, 1.5365e-01, 3.4861e-01],
        [5.2381e-01, 1.6196e-02, 1.9326e-04, 4.5980e-01],
        [2.4053e-02, 4.4341e-01, 4.9948e-01, 3.3062e-02]])
score sum of row 1 is : 1.0


In [24]:
# we can now create context aware vectors
context_aware_vectors = attention_weights @ value
print(context_aware_vectors.detach(), '\n', context_aware_vectors.shape)

tensor([[ 0.6259,  1.0413,  0.5978, -0.9497],
        [-0.3079,  0.2294,  0.3924,  0.5792],
        [ 0.3028,  1.4299,  0.5314, -0.6328],
        [-1.3347, -2.9069,  0.1679,  2.9805]]) 
 torch.Size([4, 4])
